# Bias–Variance Tradeoff

*Part of my classic ML interview preparation notes — written as a notebook so the
formulae and figures render.*

## Bias and Variance

When a model learns from data, it is trying to approximate the **true function** $f(x)$
that generates the data. But the model makes two kinds of error while doing this:

- **Bias**
  - How far off the model's prediction is from the true function.
  - Measures **systematic error** — i.e. how wrong the model is *on average*.
- **Variance**
  - How much the model's predictions fluctuate for different training sets.
  - Measures **sensitivity to noise**, or to the particular sample of data you trained on.

Therefore,

$$\text{Total Error} = \text{Bias}^2 + \text{Variance} + \underbrace{\text{Irreducible Error}}_{\text{noise in the data — can't be fixed}}$$

| Concept | In words | Analogy |
|---------|----------|---------|
| **Total Error** | How wrong your model's predictions are, on average. | The total distance of your arrows from the bullseye. |
| **Bias$^2$** | How far the *average* prediction is from the truth — systematic error. | You always aim slightly off-center. |
| **Variance** | How much your predictions jump around (instability) if you retrain on different data. | Your arrows scatter all over the target. |
| **Noise** | Randomness in the data itself (can't be fixed). | Wind gusts you can't control. |

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt

# np.polyfit warns when high-degree fits are ill-conditioned; harmless for this demo.
warnings.filterwarnings("ignore", message="Polyfit may be poorly conditioned")
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (7, 4.5)

In [ ]:
# The dartboard analogy: bullseye = truth, each dot = one model's prediction.
scenarios = [
    ("Low Bias, Low Variance", (0.0, 0.0), 0.07),
    ("Low Bias, High Variance", (0.0, 0.0), 0.30),
    ("High Bias, Low Variance", (0.45, 0.35), 0.07),
    ("High Bias, High Variance", (0.45, 0.35), 0.30),
]

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for ax, (title, center, spread) in zip(axes.ravel(), scenarios):
    for r in (1.0, 0.66, 0.33):
        ax.add_patch(plt.Circle((0, 0), r, fill=False, color="gray", lw=1))
    ax.add_patch(plt.Circle((0, 0), 0.06, color="red"))  # bullseye = true value
    pts = rng.normal(loc=center, scale=spread, size=(40, 2))
    ax.scatter(pts[:, 0], pts[:, 1], s=18, alpha=0.7, color="steelblue")
    ax.set_title(title)
    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-1.2, 1.2)
    ax.set_aspect("equal")
    ax.axis("off")
fig.suptitle("Bias vs Variance — the dartboard analogy", fontsize=13)
plt.tight_layout()
plt.show()

### Building intuition for bias

When you train a model you only ever have **one** dataset. But in theory there could have
been infinitely many datasets, each sampled from the same true distribution.

Imagine our single dataset was generated by some true function $f(x)$ that we are trying to
approximate. We can first hold out a **test split** (say 20%). Then, from the remaining
training data, we resample many times — say 10 subsets — and fit a model on each. That gives
us 10 different predictions for every point in the test set. If we **average** those 10
predictions at a point and compare that average to the point's true value, the gap is the
**bias**.

If, for most test points, the *average* prediction sits far from the true value, the models
are systematically off — they are too simple to capture the underlying pattern. That is
**high bias** (underfitting), typical of, say, fitting a straight line to curved data.

> **Don't confuse this with cross-validation.** In CV we also train on different splits, but
> we evaluate each model on its *own* held-out fold and average a **performance metric**
> (accuracy, F1, …). For the bias picture above, we instead evaluate every model on the
> *same* test set and average the **predictions** to get an average prediction per point.

### The decomposition, formally

Suppose the data is generated as

$$y = f(x) + \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0,\ \sigma^2)$$

We train our model on different datasets $D_1, D_2, \ldots, D_n$, each giving a fitted model
$\hat{f}_{D}(x)$. The **average model prediction** over all datasets is

$$\bar{f}(x) = \mathbb{E}_D\!\left[\hat{f}_D(x)\right] = \frac{1}{n}\sum_{i=1}^{n} \hat{f}_{D_i}(x)$$

i.e. the expected prediction at $x$ if we retrained on infinitely many random datasets.
The expected squared error at a point $x$ then splits cleanly into three terms:

$$\mathbb{E}_D\!\left[(y - \hat{f}_D(x))^2\right] = \underbrace{(\bar{f}(x) - f(x))^2}_{\text{Bias}^2} + \underbrace{\mathbb{E}_D\!\left[(\hat{f}_D(x) - \bar{f}(x))^2\right]}_{\text{Variance}} + \underbrace{\sigma^2}_{\text{Noise}}$$

where

- $\mathbb{E}_D$ — expectation over different datasets $D$,
- $\hat{f}_D(x)$ — model trained on dataset $D$,
- $\bar{f}(x)$ — average prediction across datasets,
- $f(x)$ — the true function,
- $\sigma^2$ — variance of the noise $\varepsilon$ (irreducible).

## The Bias–Variance Tradeoff

Bias and variance pull in opposite directions as you change model complexity:

| Model Complexity | Bias | Variance | Typical Example |
|------------------|------|----------|-----------------|
| Simple model (e.g. linear regression on nonlinear data) | High | Low | Underfitting |
| Complex model (e.g. deep neural net on a small dataset) | Low | High | Overfitting |

Total error is the **sum** of the two (plus irreducible noise), so it traces a U-shape: it
falls as we reduce bias, then rises again as variance takes over. The **sweet spot** is the
complexity that minimizes the total.

In [ ]:
# Schematic view: bias^2 falls, variance rises, total error is a U-curve.
complexity = np.linspace(1, 10, 200)
bias_sq = 3.0 / complexity        # decreases with complexity
variance = 0.05 * complexity      # increases with complexity
noise = np.full_like(complexity, 0.4)
total = bias_sq + variance + noise

plt.plot(complexity, bias_sq, label="Bias$^2$", lw=2)
plt.plot(complexity, variance, label="Variance", lw=2)
plt.plot(complexity, noise, "--", color="gray", label="Irreducible noise")
plt.plot(complexity, total, color="black", lw=2.5, label="Total error")
sweet = complexity[np.argmin(total)]
plt.axvline(sweet, color="green", ls=":", label=f"sweet spot (≈{sweet:.1f})")
plt.xlabel("Model complexity →")
plt.ylabel("Error")
plt.title("The Bias–Variance Tradeoff (schematic)")
plt.legend()
plt.tight_layout()
plt.show()

### Seeing it empirically

Let's actually *measure* the decomposition. We pick a true function
$f(x) = \sin(2\pi x)$ on a **fixed grid** of inputs, and — for each polynomial **degree**
(our complexity knob) — redraw the noisy labels many times and refit. From the resulting
cloud of predictions we estimate $\bar{f}(x)$, then compute $\text{Bias}^2$,
$\text{Variance}$, and the total. We fit with `numpy.polynomial.Polynomial.fit`, whose
scaled basis stays numerically stable even at high degree.

In [ ]:
from numpy.polynomial import Polynomial


def true_f(x):
    return np.sin(2 * np.pi * x)

sigma = 0.30          # std of the irreducible noise
n_train = 20          # fixed input grid size
n_datasets = 400      # how many times we redraw labels and refit
x_train = np.linspace(0, 1, n_train)       # same inputs every time...
x_test = np.linspace(0.02, 0.98, 100)      # ...evaluate on the interior
f_test = true_f(x_test)
degrees = range(1, 13)

bias2_by_deg, var_by_deg, err_by_deg = [], [], []
for d in degrees:
    preds = np.empty((n_datasets, x_test.size))
    for i in range(n_datasets):
        y_tr = true_f(x_train) + rng.normal(0, sigma, n_train)   # redraw noise
        preds[i] = Polynomial.fit(x_train, y_tr, d)(x_test)
    mean_pred = preds.mean(axis=0)                 # f-bar(x)
    bias2 = np.mean((mean_pred - f_test) ** 2)     # (f-bar - f)^2
    variance = np.mean(preds.var(axis=0))          # E_D[(f_hat - f-bar)^2]
    err = bias2 + variance + sigma ** 2            # expected test error
    bias2_by_deg.append(bias2)
    var_by_deg.append(variance)
    err_by_deg.append(err)

degs = np.array(list(degrees))
plt.plot(degs, bias2_by_deg, "o-", label="Bias$^2$")
plt.plot(degs, var_by_deg, "s-", label="Variance")
plt.axhline(sigma ** 2, ls="--", color="gray", label="Noise $\\sigma^2$")
plt.plot(degs, err_by_deg, "^-", color="black", label="Total error")
best = degs[int(np.argmin(err_by_deg))]
plt.axvline(best, color="green", ls=":", label=f"best degree = {best}")
plt.xlabel("Polynomial degree (model complexity)")
plt.ylabel("Error")
plt.title("Empirical bias\u2013variance decomposition")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Why 'variance' = wiggle: 40 models, each fit on the same inputs with fresh noise.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
for ax, d in zip(axes, (1, 12)):
    for _ in range(40):
        y_tr = true_f(x_train) + rng.normal(0, sigma, n_train)
        ax.plot(x_test, Polynomial.fit(x_train, y_tr, d)(x_test),
                color="steelblue", alpha=0.2)
    ax.plot(x_test, f_test, color="red", lw=2.5, label="true $f(x)$")
    label = "underfit (high bias)" if d == 1 else "overfit (high variance)"
    ax.set_title(f"degree {d}  \u2192  {label}")
    ax.set_ylim(-2, 2)
    ax.set_xlabel("x")
    ax.legend(loc="upper right")
axes[0].set_ylabel("y")
fig.suptitle("Same model class, 40 different noisy training sets", fontsize=12)
plt.tight_layout()
plt.show()

## How to move the needle

| To reduce **bias** (fix underfitting) | To reduce **variance** (fix overfitting) |
|---------------------------------------|------------------------------------------|
| Use a more expressive model / add features | Use a simpler model / drop features |
| Add polynomial or interaction terms | Gather more training data |
| Train longer; reduce regularization | Add regularization (L1/L2, dropout) |
| Reduce restrictive assumptions | Ensembling / bagging, early stopping |

## Key takeaways

- **Bias** = error from wrong assumptions (too simple) → underfitting; **variance** = error
  from sensitivity to the training sample (too complex) → overfitting.
- They **trade off**: lowering one usually raises the other, so total error is a U-curve and
  the goal is the complexity that minimizes the *sum*.
- **Irreducible noise** $\sigma^2$ sets a floor — no model can beat it.
- **More data** mainly shrinks *variance*; a **better model class / features** mainly shrinks
  *bias*.